# Mental Health LLM

## 1. Load Ollama (I would like to change my own fine tuned model later) 

In [1]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model="llama2:latest",
    temperature=0,   
)

## 2. Prompt Formatting 

<|begin_of_text|><|start_header_id|>system<|end_header_id|></br>
{system_prompt} <|eot_id|></br>
<|start_header_id|>user<|end_header_id|></br>
{user_prompt} <|eot_id|></br>
<|start_header_id|>assistant<|end_header_id|></br>

In [2]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",
         "You are a helpful assistant. Answer all questions to the best of your ability."
        ),
        ("placeholder", "{message}"),
        ("human", "{input}")  # Setting to save llm messages and User's input separately 
    ]
)

chain = prompt | llm

In [3]:
result = chain.invoke({"input": "I'm depressed"}, config={"debug": True})
print(result)

content=" Sorry to hear that you're feeling depressed. It can be a very challenging and isolating experience, but please know that you're not alone and there is help available. Depression is a common mental health condition that can affect anyone, regardless of age, gender, or background.\n\nCan you tell me more about what you're experiencing? How long have you been feeling this way, and what are some of the symptoms you're experiencing? It can be helpful to talk about your feelings and experiences with a trusted friend, family member, or mental health professional. They can offer support and help you develop strategies for managing your depression.\n\nIn the meantime, here are some things that might help:\n\n1. Reach out to a trusted friend or family member: Talking to someone you trust can help you feel less alone and more supported.\n2. Consider seeking professional help: A mental health professional can provide you with the tools and resources you need to manage your depression. Th

In [ ]:
chat1 = prompt.format_messages(input="I'm depressed")
llm.invoke(chat1)

AIMessage(content=" Sorry to hear that you're feeling depressed. It can be a very challenging and isolating experience, but please know that you're not alone and there is help available. Depression is a common mental health condition that can affect anyone, regardless of age, gender, or background.\n\nCan you tell me more about what you're experiencing? How long have you been feeling this way, and what are some of the symptoms you're experiencing? It can be helpful to talk about your feelings and experiences with a trusted friend, family member, or mental health professional. They can offer support and help you develop strategies for managing your depression.\n\nIn the meantime, here are some things that might help:\n\n1. Reach out to a trusted friend or family member: Talking to someone you trust can help you feel less alone and more supported.\n2. Consider seeking professional help: A mental health professional can provide you with the tools and resources you need to manage your depr

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}


def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory() 
    return store[session_id]

In [ ]:
chat_history_for_chain = ChatMessageHistory()

chain_with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,           # lambda session_id: chat_history_for_chain,
    input_messages_key="input",    # User's input key
    history_messages_key="message" # llm messages history key
)

print(f"chain_with_message_history: {chain_with_message_history}")
# print(f"chain_with_message_history.invoke: {chain_with_message_history.invoke({"input": "I'm depressed"})}")


chain_with_message_history: bound=RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  message: RunnableBinding(bound=RunnableLambda(_enter_history), kwargs={}, config={'run_name': 'load_history'}, config_factories=[])
}), kwargs={}, config={'run_name': 'insert_history'}, config_factories=[])
| RunnableBinding(bound=RunnableLambda(_call_runnable_sync), kwargs={}, config={'run_name': 'check_sync_or_async'}, config_factories=[]), kwargs={}, config={'run_name': 'RunnableWithMessageHistory'}, config_factories=[]) kwargs={} config={} config_factories=[] get_session_history=<function get_session_history at 0x13ae28720> input_messages_key='input' history_messages_key='message' history_factory_config=[ConfigurableFieldSpec(id='session_id', annotation=<class 'str'>, name='Session ID', description='Unique identifier for a session.', default='', is_shared=True, dependencies=None)]


In [7]:
user_input_list = []

while (True):
    user_input = input("Client's Input: ")
    user_input_list.append(user_input)
    
    print(f"Client: {user_input}")
    if user_input.lower() == "exit":
        print("Exiting the chat...")
        break
    response = chain_with_message_history.invoke(
        {"input": user_input},
        {"configurable": {"session_id": "STAR"}},
    )
    print(f"Counselor: {response}")
    print(user_input_list)

Client: I'm depressed and I have no energy to meet my friends
Counselor: content='\n Sorry to hear that you\'re feeling depressed and lacking energy to socialize with your friends. It can be really tough to deal with these feelings, but know that you\'re not alone and there are ways to cope and feel better. Here are some suggestions:\n\n1. Seek professional help: Depression is a common mental health condition that can affect anyone. If you\'re struggling with depression, consider reaching out to a mental health professional for support. They can help you identify the underlying causes of your feelings and develop strategies to manage them.\n2. Practice self-care: Take care of yourself by getting enough sleep, eating a healthy diet, and engaging in activities that bring you joy and relaxation. This can include things like reading, taking a warm bath, or practicing yoga or meditation.\n3. Connect with others: While it may be difficult to muster up the energy to socialize with friends rig

In [ ]:
storage = store["STAR"].messages
print(storage)

[HumanMessage(content="I'm depressed and I have no energy to meet my friends", additional_kwargs={}, response_metadata={}), AIMessage(content='\n Sorry to hear that you\'re feeling depressed and lacking energy to socialize with your friends. It can be really tough to deal with these feelings, but know that you\'re not alone and there are ways to cope and feel better. Here are some suggestions:\n\n1. Seek professional help: Depression is a common mental health condition that can affect anyone. If you\'re struggling with depression, consider reaching out to a mental health professional for support. They can help you identify the underlying causes of your feelings and develop strategies to manage them.\n2. Practice self-care: Take care of yourself by getting enough sleep, eating a healthy diet, and engaging in activities that bring you joy and relaxation. This can include things like reading, taking a warm bath, or practicing yoga or meditation.\n3. Connect with others: While it may be di

### Assign Score to Negative Words 

In [9]:
import re

print(user_input_list)
splt = [w for i in range(2) for w in re.findall(r"[a-zA-Z']+", user_input_list[i].lower())]
print(splt)

["I'm depressed and I have no energy to meet my friends", "I feel that I am a trash because I do nothing for future. I don't know what to do.", 'exit']
["i'm", 'depressed', 'and', 'i', 'have', 'no', 'energy', 'to', 'meet', 'my', 'friends', 'i', 'feel', 'that', 'i', 'am', 'a', 'trash', 'because', 'i', 'do', 'nothing', 'for', 'future', 'i', "don't", 'know', 'what', 'to', 'do']


In [10]:
from afinn import Afinn

def sentiment_Afinn(words):
    af = Afinn(emoticons=True)
    tuple_list = []
    for i in range(len(words)):
        iscore = af.score(words[i])

        if iscore >= 2:
            iscore = "str_pos"
        elif iscore < 2 and iscore > 0:
            iscore = "pos"
        elif iscore < 0 and iscore > -2:
            iscore = "neg"
        elif iscore <= -2:
            iscore = "str_neg"
        else:
            iscore = "mid"

        tmp = (words[i], iscore)
        tuple_list.append(tmp)
    return tuple_list    

In [11]:
data = sentiment_Afinn(splt)
print(data)

[("i'm", 'mid'), ('depressed', 'str_neg'), ('and', 'mid'), ('i', 'mid'), ('have', 'mid'), ('no', 'neg'), ('energy', 'mid'), ('to', 'mid'), ('meet', 'mid'), ('my', 'mid'), ('friends', 'mid'), ('i', 'mid'), ('feel', 'mid'), ('that', 'mid'), ('i', 'mid'), ('am', 'mid'), ('a', 'mid'), ('trash', 'mid'), ('because', 'mid'), ('i', 'mid'), ('do', 'mid'), ('nothing', 'mid'), ('for', 'mid'), ('future', 'mid'), ('i', 'mid'), ("don't", 'mid'), ('know', 'mid'), ('what', 'mid'), ('to', 'mid'), ('do', 'mid')]


In [12]:
neg = [word for word, tag in data if tag == 'neg']
print(f"neg: {neg}")

str_neg = [word for word, tag in data if tag == 'str_neg']
print(f"str_neg: {str_neg}")

neg: ['no']
str_neg: ['depressed']


### Make Report for a Doctor

In [13]:
def format_chat_history(messages):
    formatted_text = ""
    for msg in messages:
        role = "Client" if msg.type == "human" else "Counselor"
        formatted_text += f"{role}: {msg.content}\n"
    return formatted_text

full_conversation = format_chat_history(store["STAR"].messages) # If using sessions
print(full_conversation)

Client: I'm depressed and I have no energy to meet my friends
Counselor: 
 Sorry to hear that you're feeling depressed and lacking energy to socialize with your friends. It can be really tough to deal with these feelings, but know that you're not alone and there are ways to cope and feel better. Here are some suggestions:

1. Seek professional help: Depression is a common mental health condition that can affect anyone. If you're struggling with depression, consider reaching out to a mental health professional for support. They can help you identify the underlying causes of your feelings and develop strategies to manage them.
2. Practice self-care: Take care of yourself by getting enough sleep, eating a healthy diet, and engaging in activities that bring you joy and relaxation. This can include things like reading, taking a warm bath, or practicing yoga or meditation.
3. Connect with others: While it may be difficult to muster up the energy to socialize with friends right now, try to re

In [ ]:
from langchain_core.prompts import PromptTemplate

# 1. Define the Prompt
summary_prompt_template = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert psychological counselor creating a clinical summary report. 
Analyze the following conversation and produce a professional report.<|eot_id|><|start_header_id|>user<|end_header_id|>

Conversation Log:
{full_conversation}

Please provide a "Summary Report" that includes:
1. Client's Main Concerns (Key symptoms or issues mentioned)
2. Emotional State (Observed mood and sentiment)
3. Counselor's Key Advice (Summary of interventions suggested)
4. Recommended Next Steps
5. Count {neg} and {str_neg} in sentence of the Client and write them in a line


Report:<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

summary_prompt = PromptTemplate(
    template=summary_prompt_template,
    input_variables=["full_conversation"]
)

# Use 'llm' (the base pipeline) here, not 'chat_model', because we are passing a single big block of text, not a chat session.
summary_chain = summary_prompt | llm

In [15]:
print("\n--- Generating Summary Report... ---\n")

# 1. Get the full history & Summarize it
conversation_text = format_chat_history(chat_history_for_chain.messages)
report = summary_chain.invoke({"conversation": conversation_text})

# print(report)
print(report.content)


--- Generating Summary Report... ---

Summary Report:

1. Client's Main Concerns:
	* Feeling depressed and lacking energy to socialize with friends
	* Feeling purposeless and unsure of how to improve their future
2. Emotional State:
	* Observed mood: Depressed
	* Sentiment: Uncertain, lost, and hopeless
3. Counselor's Key Advice:
	* Seek professional help for depression
	* Practice self-care and engage in activities that bring joy and relaxation
	* Connect with others and talk through feelings with a trusted friend or therapist
	* Try grounding techniques, mindfulness, and new hobbies to find purpose and fulfillment
	* Set realistic goals and take small steps towards them
4. Recommended Next Steps:
	* Schedule an appointment with a mental health professional for further support
	* Practice self-compassion and be patient with oneself as progress may not always be linear
	* Try new activities or hobbies to discover new passions and interests
5. Count ['no'] and ['depressed'] in sentence